# Milestone 2: Data Cleaning IDN News

Notebook ini melanjutkan `01_data_profiling.ipynb` untuk dataset berita Indonesia.

## Tujuan
1. Memeriksa kualitas data awal.
2. Menstandarkan teks dan mengonversi tanggal dengan aman.
3. Menghapus duplikasi identik pada `date_raw`, `link`, `title`, dan `text`.
4. Mempertahankan record dengan URL sama jika atribut lainnya berbeda.
5. Menyimpan hasil cleaning terpisah dari dataset mentah.
6. Memvalidasi hasil menggunakan DuckDB `SUMMARIZE` dan menyimpan laporan kualitas data.

**Catatan pemeriksaan awal:** 441 file Parquet, 1.149.789 baris, ukuran sekitar 1,558 GB, dan 101.421 nilai teks `NaT` pada kolom tanggal. Nilai tersebut tidak otomatis menyebabkan berita dihapus.

Jalankan notebook dari direktori utama repository `tugas1-reyboy11`. Dataset mentah tidak ditimpa.

## 1. Import library dan konfigurasi

Notebook menggunakan Polars untuk pemrosesan data dan DuckDB untuk profiling akhir.

In [4]:

from pathlib import Path
import json
import polars as pl
import duckdb

# Deteksi root proyek secara otomatis
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    ROOT = CURRENT_DIR.parent
else:
    ROOT = CURRENT_DIR

DATA_GLOB = ROOT / "data" / "raw" / "idn-news" / "data_files" / "*.parquet"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "output" / "tables"

CLEANED_PATH = PROCESSED_DIR / "idn_news_cleaned.parquet"
QUALITY_REPORT_PATH = OUTPUT_DIR / "data_quality_report.json"
SUMMARIZE_PATH = OUTPUT_DIR / "duckdb_summarize.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

parquet_files = sorted(DATA_GLOB.parent.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"Dataset tidak ditemukan. Lokasi yang diperiksa: {DATA_GLOB}"
    )

print("Root proyek:", ROOT)
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)
print("Jumlah file Parquet:", len(parquet_files))
print(
    "Total ukuran (GB):",
    round(sum(p.stat().st_size for p in parquet_files) / 1e9, 3)
)
print("Lokasi output:", CLEANED_PATH)


Root proyek: C:\Users\raihan bhakti\Desktop\tugas1-reyboy11
Polars: 1.44.2
DuckDB: 1.5.5
Jumlah file Parquet: 441
Total ukuran (GB): 1.558
Lokasi output: C:\Users\raihan bhakti\Desktop\tugas1-reyboy11\data\processed\idn_news_cleaned.parquet


## 2. Membaca dataset dan memeriksa skema

`scan_parquet` digunakan agar Polars membaca data secara lazy. File mentah tidak dimodifikasi.

In [5]:
raw = pl.scan_parquet(str(DATA_GLOB))
print(raw.collect_schema())

Schema([('date', String), ('link', String), ('title', String), ('text', String)])


## 3. Profiling kualitas data sebelum cleaning

Angka dihitung ulang dari dataset lokal saat notebook dijalankan.

In [6]:
date_parsed_expr = pl.col("date").str.strip_chars().str.to_date(strict=False)

baseline = raw.select(
    pl.len().alias("total_baris_awal"),
    pl.col("date").null_count().alias("date_null"),
    (pl.col("date").is_not_null() & date_parsed_expr.is_null()).sum().alias("date_gagal_parsing"),
    (pl.col("date").str.strip_chars().str.to_uppercase() == "NAT").sum().alias("date_teks_nat"),
    pl.col("link").null_count().alias("link_null"),
    (pl.col("link").str.strip_chars() == "").sum().alias("link_kosong"),
    pl.col("title").null_count().alias("title_null"),
    (pl.col("title").str.strip_chars() == "").sum().alias("title_kosong"),
    pl.col("text").null_count().alias("text_null"),
    (pl.col("text").str.strip_chars() == "").sum().alias("text_kosong"),
    (pl.len() - pl.col("link").n_unique()).alias("kelebihan_baris_url_berulang"),
    (pl.len() - pl.struct(["date", "link", "title", "text"]).n_unique()).alias("kelebihan_baris_identik_awal"),
).collect(engine="streaming")

display(baseline)

total_baris_awal,date_null,date_gagal_parsing,date_teks_nat,link_null,link_kosong,title_null,title_kosong,text_null,text_kosong,kelebihan_baris_url_berulang,kelebihan_baris_identik_awal
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1149789,0,101421,101421,0,0,0,0,0,0,120304,108391


## 4. Aturan cleaning

- Pertahankan tanggal asli sebagai `date_raw`.
- Ubah teks `NaT` menjadi null pada kolom hasil konversi `date`; record tetap dipertahankan.
- Pangkas spasi awal/akhir pada `link`, `title`, dan `text`.
- Keluarkan baris yang `link`, `title`, atau `text`-nya null/kosong.
- Hapus hanya duplikasi identik pada `date_raw`, `link`, `title`, dan `text`.
- Jangan menghapus semua record dengan URL sama. URL sama dapat memiliki tanggal atau judul berbeda.

Hasil disimpan ke `data/processed/idn_news_cleaned.parquet`; dataset mentah tidak ditimpa.

In [8]:

prepared = (
    raw.rename({"date": "date_raw"})
    .with_columns(
        pl.col("date_raw").str.strip_chars().alias("date_raw"),
        pl.col("link").str.strip_chars().alias("link"),
        pl.col("title").str.strip_chars().alias("title"),
        pl.col("text").str.strip_chars().alias("text"),
    )
    .with_columns(
        pl.when(
            pl.col("date_raw").str.to_uppercase() == "NAT"
        )
        .then(None)
        .otherwise(pl.col("date_raw"))
        .str.to_date(strict=False)
        .alias("date")
    )
    .filter(
        pl.col("link").is_not_null()
        & (pl.col("link") != "")
        & pl.col("title").is_not_null()
        & (pl.col("title") != "")
        & pl.col("text").is_not_null()
        & (pl.col("text") != "")
    )
    .select("date", "date_raw", "link", "title", "text")
)

# Hitung tanggal valid yang berada di luar periode dataset.
out_of_range_rows = int(
    prepared.filter(
        pl.col("date").is_not_null()
        & (
            (pl.col("date") < pl.date(2022, 1, 1))
            | (pl.col("date") > pl.date(2023, 10, 31))
        )
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

# Pertahankan tanggal null, tetapi keluarkan tanggal valid di luar periode.
prepared = prepared.filter(
    pl.col("date").is_null()
    | (
        (pl.col("date") >= pl.date(2022, 1, 1))
        & (pl.col("date") <= pl.date(2023, 10, 31))
    )
)

print("Baris bertanggal di luar periode:", out_of_range_rows)
print(
    "Baris setelah validasi kolom dan periode:",
    prepared.select(pl.len()).collect(engine="streaming").item()
)
print("Skema hasil:")
print(prepared.collect_schema())


Baris bertanggal di luar periode: 4
Baris setelah validasi kolom dan periode: 1149785
Skema hasil:
Schema([('date', Date), ('date_raw', String), ('link', String), ('title', String), ('text', String)])


## 5. Menghapus duplikasi identik dan menyimpan hasil

Deduplikasi memakai keempat kolom sumber. Record dengan URL sama tetapi tanggal atau judul berbeda tidak otomatis dihapus.

In [9]:
prepared_rows = int(
    prepared.select(pl.len().alias("n")).collect(engine="streaming").item()
)

deduplicated = prepared.unique(
    subset=["date_raw", "link", "title", "text"],
    keep="first",
    maintain_order=False,
)

deduplicated.sink_parquet(
    str(CLEANED_PATH),
    compression="zstd",
    maintain_order=False,
    engine="streaming",
)

cleaned = pl.scan_parquet(str(CLEANED_PATH))
cleaned_rows = int(
    cleaned.select(pl.len().alias("n")).collect(engine="streaming").item()
)

print(f"Baris setelah validasi kolom wajib: {prepared_rows:,}")
print(f"Baris setelah deduplikasi identik: {cleaned_rows:,}")
print(f"Baris duplikat identik yang dihilangkan: {prepared_rows - cleaned_rows:,}")
print(f"File hasil: {CLEANED_PATH}")
print(f"Ukuran hasil (MB): {CLEANED_PATH.stat().st_size / 1e6:.2f}")

Baris setelah validasi kolom wajib: 1,149,785
Baris setelah deduplikasi identik: 1,041,394
Baris duplikat identik yang dihilangkan: 108,391
File hasil: C:\Users\raihan bhakti\Desktop\tugas1-reyboy11\data\processed\idn_news_cleaned.parquet
Ukuran hasil (MB): 889.61


## 6. Validasi kualitas data setelah cleaning

Pastikan file hasil dapat dibaca dan tidak ada duplikasi identik yang tersisa berdasarkan aturan yang digunakan.

In [10]:
post_validation = cleaned.select(
    pl.len().alias("total_baris_hasil"),
    pl.col("date").null_count().alias("date_null_hasil"),
    pl.col("date_raw").null_count().alias("date_raw_null"),
    (pl.col("date_raw").str.to_uppercase() == "NAT").sum().alias("date_raw_nat"),
    pl.col("link").null_count().alias("link_null_hasil"),
    (pl.col("link").str.strip_chars() == "").sum().alias("link_kosong_hasil"),
    pl.col("title").null_count().alias("title_null_hasil"),
    (pl.col("title").str.strip_chars() == "").sum().alias("title_kosong_hasil"),
    pl.col("text").null_count().alias("text_null_hasil"),
    (pl.col("text").str.strip_chars() == "").sum().alias("text_kosong_hasil"),
    (pl.len() - pl.struct(["date_raw", "link", "title", "text"]).n_unique()).alias("kelebihan_baris_identik_hasil"),
).collect(engine="streaming")

display(post_validation)

total_baris_hasil,date_null_hasil,date_raw_null,date_raw_nat,link_null_hasil,link_kosong_hasil,title_null_hasil,title_kosong_hasil,text_null_hasil,text_kosong_hasil,kelebihan_baris_identik_hasil
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1041394,101403,0,101403,0,0,0,0,0,0,0


## 7. Profiling akhir menggunakan DuckDB `SUMMARIZE`

DuckDB membaca file hasil cleaning. Ringkasan disimpan sebagai CSV untuk dokumentasi.

In [11]:
con = duckdb.connect(database=":memory:")
cleaned_path_sql = CLEANED_PATH.resolve().as_posix().replace("'", "''")

summarize_df = con.execute(
    f"SUMMARIZE SELECT * FROM read_parquet('{cleaned_path_sql}')"
).df()

display(summarize_df)
summarize_df.to_csv(SUMMARIZE_PATH, index=False)
print(f"Ringkasan DuckDB disimpan ke: {SUMMARIZE_PATH}")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,date,DATE,2022-03-28,2023-10-01,524,2023-05-03 22:35:37.187271,<NA>,2023-02-16,2023-05-20,2023-07-28,1041394,9.74
1,date_raw,VARCHAR,2022-03-28,NaT,379,None,<NA>,None,None,None,1041394,0.00
2,link,VARCHAR,https://20.detik.com/blak-blakan/20230620-2306...,https://yogyakarta.kompas.com/read/2023/10/01/...,949847,None,<NA>,None,None,None,1041394,0.00
3,title,VARCHAR,"""100% Manusia Film Festival 2023"" siap putar 6...","﻿Unsur-unsur Surat Pribadi, Lengkap dengan Cir...",851378,None,<NA>,None,None,None,1041394,0.00
4,text,VARCHAR,"""\n\nJakarta (ANTARA) - Ketua ASEAN Business A...",• Perusahaan Berkomitmen untuk Memaksimalkan N...,817458,None,<NA>,None,None,None,1041394,0.00


Ringkasan DuckDB disimpan ke: C:\Users\raihan bhakti\Desktop\tugas1-reyboy11\output\tables\duckdb_summarize.csv


## 8. Menyimpan laporan kualitas data dalam JSON

Laporan berisi angka aktual dari eksekusi notebook, bukan angka yang ditulis manual.

In [13]:

# Perbarui laporan agar sesuai dengan proses cleaning terbaru.

report["cleaning_rules"].append(
    "Exclude records with valid dates outside 2022-01-01 to 2023-10-31."
)

report["date_range_filter"] = {
    "start_date": "2022-01-01",
    "end_date": "2023-10-31",
    "rows_excluded": int(out_of_range_rows)
}

report["rows_excluded_outside_date_range"] = int(out_of_range_rows)

# Perjelas bahwa jumlah ini sudah melewati validasi
# kolom wajib dan filter rentang tanggal.
report[
    "rows_after_required_field_and_date_range_validation_before_deduplication"
] = int(prepared_rows)

report.pop(
    "rows_after_required_field_validation_before_deduplication",
    None
)

with open(QUALITY_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print("Laporan kualitas data berhasil diperbarui.")
print("Baris di luar periode:", report["rows_excluded_outside_date_range"])
print("Total baris setelah cleaning:", report["rows_after_cleaning"])
print("Lokasi laporan:", QUALITY_REPORT_PATH)


Laporan kualitas data berhasil diperbarui.
Baris di luar periode: 4
Total baris setelah cleaning: 1041394
Lokasi laporan: C:\Users\raihan bhakti\Desktop\tugas1-reyboy11\output\tables\data_quality_report.json


## 9. Kesimpulan sementara

Lengkapi kesimpulan berdasarkan angka aktual yang muncul setelah notebook dijalankan. Bahas jumlah baris awal dan akhir, duplikasi identik yang dihapus, tanggal yang tidak dapat diparsing, dan hasil validasi DuckDB.

Notebook ini hanya menghapus record yang identik berdasarkan `date_raw`, `link`, `title`, dan `text`. Record dengan URL sama tetapi atribut berbeda tetap dipertahankan untuk menghindari kehilangan informasi tanpa dasar yang cukup.

## Referensi dataset

README dataset lokal mencantumkan lisensi CC BY 4.0 dan menjelaskan bahwa korpus dikumpulkan dari sejumlah portal berita Indonesia untuk periode Januari hingga Oktober 2023, dengan beberapa artikel dari 2022. Pertahankan atribusi kepada penyedia dataset saat mendistribusikan hasil.